# Comprehensive Evaluation and Comparison

## Objective
Consolidate and compare all anomaly detection methods across three paradigms:
1. **Unsupervised:** Isolation Forest, LOF, One-Class SVM
2. **Supervised:** Random Forest, XGBoost, Gradient Boosting, Voting Ensemble
3. **Deep Learning:** Dense Autoencoder, LSTM Autoencoder

Evaluation uses precision, recall, F1-score, MCC, ROC-AUC, and Average Precision — metrics specifically suited for imbalanced anomaly detection tasks.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.evaluation import plot_model_comparison

DATA_DIR = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results', 'figures')

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
print('Libraries loaded')

## 7.1 Load All Results

In [ ]:
unsup_df = pd.read_csv(os.path.join(DATA_DIR, 'unsupervised_results.csv'), index_col='model')
sup_df = pd.read_csv(os.path.join(DATA_DIR, 'plant_supervised_results.csv'), index_col='model')
ae_df = pd.read_csv(os.path.join(DATA_DIR, 'autoencoder_results.csv'), index_col='model')

# Combine all
all_results = pd.concat([unsup_df, sup_df, ae_df])
print('Combined results:')
metric_cols = ['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'avg_precision', 'train_time', 'type']
available = [c for c in metric_cols if c in all_results.columns]
print(all_results[available].round(4).to_string())

## 7.2 Overall Performance Comparison

In [ ]:
plot_model_comparison(
    all_results,
    metric_cols=['precision', 'recall', 'f1_score', 'mcc'],
    save_path=os.path.join(RESULTS_DIR, 'all_models_comparison.png')
)

## 7.3 Paradigm Comparison (Unsupervised vs Supervised vs Deep Learning)

In [ ]:
if 'type' in all_results.columns:
    paradigm_summary = all_results.groupby('type')[['precision', 'recall', 'f1_score', 'mcc']].agg(['mean', 'max'])
    print('Performance by Paradigm:')
    print(paradigm_summary.round(4).to_string())
    
    # Best model per paradigm
    print('\nBest model per paradigm (by F1-score):')
    for ptype in all_results['type'].unique():
        subset = all_results[all_results['type'] == ptype]
        best = subset['f1_score'].idxmax()
        print(f'  {ptype:15s}: {best} (F1={subset.loc[best, "f1_score"]:.4f})')

In [ ]:
# Radar chart for best model per paradigm
if 'type' in all_results.columns:
    metrics_for_radar = ['precision', 'recall', 'f1_score', 'mcc']
    available_radar = [m for m in metrics_for_radar if m in all_results.columns]
    
    best_models = {}
    for ptype in all_results['type'].unique():
        subset = all_results[all_results['type'] == ptype]
        best_name = subset['f1_score'].idxmax()
        best_models[f'{best_name} ({ptype})'] = subset.loc[best_name, available_radar].fillna(0).values
    
    angles = np.linspace(0, 2 * np.pi, len(available_radar), endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = plt.cm.Set2(np.linspace(0, 1, len(best_models)))
    
    for (name, values), color in zip(best_models.items(), colors):
        vals = values.tolist() + [values[0]]
        ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=color)
        ax.fill(angles, vals, alpha=0.15, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m.replace('_', ' ').title() for m in available_radar])
    ax.set_ylim(0, 1.05)
    ax.set_title('Best Model Per Paradigm — Radar Comparison', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'radar_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 7.4 Training Time Analysis

In [ ]:
if 'train_time' in all_results.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    times = all_results['train_time'].sort_values()
    colors_time = ['#2ecc71' if t < 5 else '#f39c12' if t < 30 else '#e74c3c' for t in times.values]
    times.plot(kind='barh', ax=ax, color=colors_time, edgecolor='black')
    ax.set_xlabel('Training Time (seconds)')
    ax.set_title('Training Time Comparison')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'training_times.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 7.5 UCI Grid Stability Results

In [ ]:
uci_sup_df = pd.read_csv(os.path.join(DATA_DIR, 'uci_supervised_results.csv'), index_col='model')
print('UCI Grid Stability — Supervised Results:')
uci_metric_cols = [c for c in ['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'train_time'] if c in uci_sup_df.columns]
print(uci_sup_df[uci_metric_cols].round(4).to_string())

## 7.6 Final Summary

In [ ]:
print('='*70)
print('FINAL RESULTS SUMMARY')
print('='*70)

print('\n--- Power Plant Dataset (All Methods) ---')
summary_cols = [c for c in ['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'type'] if c in all_results.columns]
print(all_results[summary_cols].sort_values('f1_score', ascending=False).round(4).to_string())

print(f'\nOverall best model: {all_results["f1_score"].idxmax()} '
      f'(F1={all_results["f1_score"].max():.4f})')

print('\n--- UCI Grid Stability (Supervised Only) ---')
uci_cols = [c for c in ['precision', 'recall', 'f1_score', 'mcc', 'roc_auc'] if c in uci_sup_df.columns]
print(uci_sup_df[uci_cols].sort_values('f1_score', ascending=False).round(4).to_string())

## Conclusion

This study presented a comprehensive evaluation of anomaly detection methods for smart power grid systems, comparing unsupervised, supervised, and deep learning approaches across two datasets.

### Key Findings

1. **Supervised methods outperform unsupervised** when labeled data is available, with ensemble methods (Voting, XGBoost) achieving the highest F1-scores
2. **LSTM Autoencoders** capture temporal fault signatures that Dense Autoencoders miss, particularly for sensor drift anomalies
3. **Feature engineering is critical** — domain-specific features (power factor deviation, thermal efficiency, interaction terms) dramatically improve detection across all model families
4. **Isolation Forest** is the strongest unsupervised method, providing reasonable detection without any labeled anomaly data
5. **MCC is the most informative metric** for imbalanced anomaly detection, as it accounts for all quadrants of the confusion matrix

### Practical Implications

- For **real-time deployment** with no historical fault data: use Isolation Forest or LSTM Autoencoder
- For **offline analysis** with labeled incidents: use XGBoost or Voting Ensemble with engineered features
- **Feature engineering** based on electrical engineering domain knowledge provides more value than model complexity alone